# Phân tích khám phá dữ liệu (EDA) thị trường cho thuê RoomBeacon

Notebook này bao gồm các phần sau:

- [Nhập các thư viện](#nhập-các-thư-viện)

- [Kết nối DuckDB và tải dataset](#kết-nối-duckdb-và-tải-dataset)

- [Tổng quan về dataset](#tổng-quan-về-dataset)

- [Đánh giá chất lượng dữ liệu](#data-quality-assessment)

- [Phân tích nguồn dữ liệu](#phân-tích-nguồn-dữ-liệu)

- [Phân tích giá thuê](#phân-tích-giá-thuê)

- [Phân tích diện tích cho thuê](#phân-tích-diện-tích-cho-thuê)

- [Phát hiện giá trị ngoại lệ](#phát-hiện-giá-trị-ngoại-lệ)

- [Phân tích mối quan hệ](#phân-tích-mối-quan-hệ)

- [Phân tích theo thời gian](#phân-tích-theo-thời-gian)

- [Tóm tắt insight dữ liệu](#tóm-tắt-insight-dữ-liệu)

## Nhập các thư viện
Phần này nhập các thư viện Python cần thiết để xử lý dữ liệu, trực quan hóa và thực hiện phân tích khám phá dữ liệu.

In [19]:
try:
    from utils import setup_project_path
except ModuleNotFoundError:
    from notebooks.utils import setup_project_path

PROJECT_ROOT = setup_project_path()

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=False)

import numpy as np
import pandas as pd
from analytics.duckdb.connection import create_analytics_connection

print("Đã nhập thư viện và thiết lập môi trường thành công.")

Đã phát hiện thư mục gốc của dự án RoomBeacon: /mnt/Data/projects/roombeacon
Đã nhập thư viện và thiết lập môi trường thành công.


## Kết nối DuckDB và tải dataset

Phần này kết nối với database DuckDB của RoomBeacon và tải dataset phân tích vào Pandas DataFrame để thực hiện phân tích khám phá.

In [20]:
# Factory của dự án tạo catalog DuckDB, gắn Bronze MySQL bằng cú pháp DSN
# key=value tương thích với DuckDB và khởi tạo các analytical view.
conn = create_analytics_connection()

attached_databases = {row[0] for row in conn.execute("SHOW DATABASES").fetchall()}
if "mysql_db" not in attached_databases:
    raise RuntimeError(
        "DuckDB không thể gắn Bronze MySQL. Hãy bảo đảm mysql-bronze đang chạy "
        "và thông tin kết nối BRONZE_MYSQL_HOST/PORT trong .env có thể truy cập "
        "từ môi trường notebook hiện tại."
    )

In [21]:
print("Đã kết nối thành công!")
conn.sql("SHOW TABLES;").show()

Đã kết nối thành công!
┌──────────────────────────┐
│           name           │
│         varchar          │
├──────────────────────────┤
│ v_acquisition_efficiency │
│ v_content_changes        │
│ v_data_quality           │
│ v_latest_posts           │
│ v_listing_lifetime       │
│ v_location_summary       │
│ v_observations           │
│ v_price_history          │
│ v_source_activity        │
└──────────────────────────┘



In [22]:
df = conn.sql("SELECT * FROM v_latest_posts").df()
type(df)

pandas.DataFrame

## Tổng quan về dataset

Phần này cung cấp cái nhìn tổng quan về dataset được sử dụng trong quá trình EDA.

Các mục tiêu chính bao gồm:

- Kiểm tra kích thước tổng thể của dataset.
- Xác nhận cấu trúc và schema của dữ liệu.
- Kiểm tra thông tin các trường dữ liệu.
- Quan sát một số bản ghi mẫu.
- Đánh giá sơ bộ các biến số quan trọng trước khi thực hiện phân tích chuyên sâu.

Các nội dung thực hiện:

- Kích thước dataset:
    - Số lượng bản ghi.
    - Số lượng đặc trưng.

- Xem trước dataset:
    - Quan sát dữ liệu mẫu.
    - Kiểm tra định dạng và giá trị ban đầu.

- Thông tin dataset:
    - Tên các cột.
    - Kiểu dữ liệu.
    - Số lượng giá trị không null.

- Thống kê cơ bản:
    - Thống kê mô tả các biến số.
    - Kiểm tra phạm vi giá trị của các trường dạng số.

Kết quả của phần này giúp xác định cấu trúc dataset và chuẩn bị cho các bước tiếp theo như đánh giá chất lượng dữ liệu và phân tích khám phá.

### Kích thước dataset
Kiểm tra kích thước tổng thể của dataset bao gồm:

- Số lượng bản ghi (hàng).
- Số lượng đặc trưng (cột).

In [23]:
df.shape

(55124, 14)

In [24]:
df.columns

Index(['source_code', 'rental_post_id', 'source_listing_id', 'title_raw',
       'url', 'price_amount', 'area_value', 'full_address_text',
       'location_raw', 'full_address_inherited', 'latest_observed_at',
       'first_observed_at', 'last_observed_at', 'active_days'],
      dtype='str')

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 55124 entries, 0 to 55123
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   source_code             55124 non-null  str           
 1   rental_post_id          55124 non-null  int64         
 2   source_listing_id       55124 non-null  str           
 3   title_raw               54862 non-null  str           
 4   url                     55124 non-null  str           
 5   price_amount            46108 non-null  float64       
 6   area_value              43892 non-null  float64       
 7   full_address_text       12019 non-null  str           
 8   location_raw            12019 non-null  str           
 9   full_address_inherited  55124 non-null  bool          
 10  latest_observed_at      55124 non-null  datetime64[us]
 11  first_observed_at       55124 non-null  datetime64[us]
 12  last_observed_at        55124 non-null  datetime64[us]
 1

#### Chú thích

Dataset `v_latest_posts` gồm 14 trường dữ liệu chính, đại diện cho thông tin định danh, thuộc tính, vị trí và vòng đời của tin đăng cho thuê.

Các nhóm trường chính:

### Thông tin định danh và nguồn dữ liệu

- `source_code`: Nguồn dữ liệu được crawl, xác định nền tảng đăng tin.
- `rental_post_id`: ID nội bộ duy nhất của RoomBeacon cho mỗi tin đăng.
- `source_listing_id`: ID gốc của tin đăng trên trang web nguồn.
- `url`: Đường dẫn truy cập tin đăng gốc.

### Thuộc tính tin đăng

- `title_raw`: Tiêu đề nguyên bản của tin đăng.
- `price_amount`: Giá thuê được bóc tách dạng số (VNĐ/tháng).
- `area_value`: Diện tích tin đăng (m²).

### Thông tin vị trí

- `full_address_text`: Địa chỉ chi tiết thu thập từ nguồn.
- `location_raw`: Trường địa chỉ thô phục vụ phân tích vị trí.
- `full_address_inherited`: Đánh dấu địa chỉ được lấy trực tiếp hay kế thừa từ lịch sử quan sát.

### Thời gian và vòng đời

- `latest_observed_at`: Thời điểm crawler quan sát tin gần nhất.
- `first_observed_at`: Thời điểm đầu tiên phát hiện tin đăng.
- `last_observed_at`: Thời điểm cuối cùng ghi nhận hoạt động.
- `active_days`: Số ngày tin đăng tồn tại trên hệ thống.

Các trường dữ liệu này phục vụ cho các bước phân tích tiếp theo như:
- Đánh giá chất lượng dữ liệu.
- Phân tích giá thuê và diện tích.
- Phân tích nguồn dữ liệu.
- Nghiên cứu vòng đời tin đăng.

### Xem trước dataset
- Quan sát dữ liệu mẫu.
- Kiểm tra định dạng và giá trị ban đầu.


In [26]:
df.head(5)

,source_code,rental_post_id,source_listing_id,title_raw,url,price_amount,area_value,full_address_text,location_raw,full_address_inherited,latest_observed_at,first_observed_at,last_observed_at,active_days
0,chothuephongtro,776381,164748,Cho Thuê Studio 29m2 Ban Công Thoáng - Gần Học...,https://chothuephongtro.me/cho-thue-studio-29m...,15000000.0,29.0,"6 Đồ Sơn, Phường 2, Quận Tân Bình","6 Đồ Sơn, Phường 2, Quận Tân Bình",False,2026-08-29 01:22:11,2026-08-27 16:35:40,2026-08-29 01:22:11,2
1,chothuephongtro,776380,164773,STUDIO CỬA SỔ LỚN – FULL NỘI THẤT – GẦN CÔNG V...,https://chothuephongtro.me/studio-cua-so-lon-f...,15000000.0,45.0,"Phổ Quang, P2, Quận Tân Bình Gần Công viên Gia...","Phổ Quang, P2, Quận Tân Bình Gần Công viên Gia...",False,2026-08-29 01:22:09,2026-08-27 16:35:40,2026-08-29 01:22:09,2
2,chothuephongtro,776379,164774,Phòng Trọ Gía sinh viên chỉ 3tr ở P13 Quận Tân...,https://chothuephongtro.me/phong-tro-gia-sinh-...,15000000.0,25.0,"Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...","Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...",False,2026-08-29 01:22:07,2026-08-27 16:35:40,2026-08-29 01:22:07,2
3,chothuephongtro,776378,164776,Phòng trọ có gác ở được 4 người gần ĐH Văn Hiến,https://chothuephongtro.me/phong-tro-co-gac-o-...,15000000.0,25.0,"Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...","Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...",False,2026-08-29 01:22:05,2026-08-27 16:35:40,2026-08-29 01:22:05,2
4,chothuephongtro,776377,164778,"DUPLEX BAN CÔNG MÁY GIẶT RIÊNG. Gần ĐH UFM, Vi...",https://chothuephongtro.me/duplex-ban-cong-may...,15000000.0,35.0,"Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...","Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...",False,2026-08-29 01:22:03,2026-08-27 16:35:40,2026-08-29 01:22:03,2


In [27]:
# Kiểm tra số bản ghi có full_address_inherited = true
address_inherited = conn.query("SELECT COUNT(*) FROM v_latest_posts WHERE full_address_inherited = 1")
print(address_inherited)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1309 │
└──────────────┘



In [28]:
# Tổng quan về các đặc trưng dạng số
# Khảo sát phân phối thống kê của các trường dạng số quan trọng:
# price_amount, area_value và active_days.

overview_features = [
    "rental_post_id",
    "price_amount",
    "area_value",
    "active_days",
    "full_address_text",
    "location_raw"
]

df[overview_features].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
rental_post_id,55124.0,NaN,NaN,NaN,801242.909277,26066.599698,757140.0,779790.75,799288.5,823555.25,847829.0
price_amount,46108.0,NaN,NaN,NaN,5237912.827774,43916210.192574,3500.0,3000000.0,4500000.0,6000000.0,7777000000.0
area_value,43892.0,NaN,NaN,NaN,33.196895,30.450145,1.0,25.0,30.0,35.0,2300.0
active_days,55124.0,NaN,NaN,NaN,0.290491,0.705075,0.0,0.0,0.0,0.0,5.0
full_address_text,12019,5033,"Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...",1812,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_raw,12019,5033,"Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...",1812,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Tổng quan đặc trưng dataset

Thống kê mô tả cung cấp góc nhìn ban đầu về số lượng giá trị, phân phối và phạm vi của các trường quan trọng. Những quan sát này là cơ sở để xác định nội dung cần kiểm tra sâu hơn, chưa phải kết luận cuối cùng về chất lượng dữ liệu.

## Data Quality Assessment

Phần này đánh giá chất lượng dữ liệu trước khi thực hiện cleaning, tập trung vào missing values, duplicate và các vấn đề validation. Các phát hiện sẽ được sử dụng để xây dựng cleaning rules phù hợp cho dataset RoomBeacon.

### Missing Value Analysis

**Mục tiêu:**

- Kiểm tra mức độ thiếu dữ liệu của các trường quan trọng.
- Xác định các field cần được điều tra thêm trước khi xây dựng cleaning rules.

Missing value chưa đồng nghĩa với lỗi dữ liệu. Với `price_amount`, giá trị `NULL` có thể xuất hiện khi website không cung cấp giá số, listing hiển thị "Thỏa thuận", parser chưa trích xuất được dữ liệu hoặc các source sử dụng format khác nhau. Cần kiểm tra bằng chứng từ dữ liệu gốc trước khi kết luận nguyên nhân.

### Missing Price Investigation

Biểu thức `missing_price = df[df["price_amount"].isna()]` lọc các listing chưa có giá số để phục vụ điều tra. Việc lấy thêm `source_code`, `url` và `title_raw` giúp:

- Truy ngược nguồn dữ liệu.
- Đối chiếu nội dung thực tế trên website.
- Xác định nguyên nhân missing dựa trên bằng chứng thay vì suy đoán.

In [29]:
# Kiểm tra các giá trị NaN
missing_price = df[df["price_amount"].isna()]
missing_price[
    [
        "source_code",
        "url",
        "title_raw"
    ]
].head()

,source_code,url,title_raw
74,chothuephongtro,https://chothuephongtro.me/ho-chi-minh/go-vap/...,CẦN CHO THUÊ STUDIO NGAY CẦU AN LỘC QUẬN GÒ VẤP.
81,chothuephongtro,https://chothuephongtro.me/ho-chi-minh/phu-nhu...,CHDV BAN CÔNG VÀ CỬA SỔ QUẬN PHÚ NHUẬN
103,chothuephongtro,https://chothuephongtro.me/ho-chi-minh/quan-8/...,Khai trương căn hộ ban công thoáng mát new 100...
114,chothuephongtro,https://chothuephongtro.me/ho-chi-minh/quan-7/...,Cho thuê CHDV_full nt _1PN NGAY Trần xuân Soạn...
138,chothuephongtro,https://chothuephongtro.me/ho-chi-minh/go-vap/...,"CẦN CHO THUÊ DUPLEX NGAY CAO ĐẲNG NOVA, ĐẠI HỌ..."


In [30]:
df["source_code"].value_counts()

source_code
chothuephongtro    23089
mogi               17402
cafeland            9991
nhatot              1550
phongtro123         1120
tromoi              1019
nhatrovn             877
chothuenha            76
Name: count, dtype: int64

# Data Quality Assessment

Phần này đánh giá chất lượng dữ liệu hiện tại trước khi thực hiện các bước làm sạch và chuẩn hóa.

Mục tiêu của bước này là xác định các vấn đề có thể ảnh hưởng đến quá trình phân tích và xây dựng các quy tắc xử lý dữ liệu phù hợp.

Các kiểm tra chính bao gồm:

- **Missing Values**: Đánh giá mức độ thiếu dữ liệu của các trường quan trọng như giá thuê, diện tích và thông tin vị trí.
- **Data Validation**: Kiểm tra tính hợp lệ của dữ liệu như kiểu dữ liệu, khoảng giá trị và các ràng buộc nghiệp vụ.
- **Duplicate Check**: Phát hiện các bản ghi trùng lặp trong dataset.
- **Location Validation**: Kiểm tra và chuẩn hóa thông tin địa chỉ theo cấu trúc hành chính.

Các vấn đề phát hiện trong bước này sẽ được sử dụng để xây dựng quy trình cleaning và transformation trước khi tạo dataset phục vụ cho các bước phân tích tiếp theo.

In [31]:
df.dtypes

source_code                          str
rental_post_id                     int64
source_listing_id                    str
title_raw                            str
url                                  str
price_amount                     float64
area_value                       float64
full_address_text                    str
location_raw                         str
full_address_inherited              bool
latest_observed_at        datetime64[us]
first_observed_at         datetime64[us]
last_observed_at          datetime64[us]
active_days                        int64
dtype: object

In [32]:
df.isnull().sum()

source_code                   0
rental_post_id                0
source_listing_id             0
title_raw                   262
url                           0
price_amount               9016
area_value                11232
full_address_text         43105
location_raw              43105
full_address_inherited        0
latest_observed_at            0
first_observed_at             0
last_observed_at              0
active_days                   0
dtype: int64

In [33]:
missing_rate = (
    df.isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

missing_rate

location_raw              78.20
full_address_text         78.20
area_value                20.38
price_amount              16.36
title_raw                  0.48
rental_post_id             0.00
url                        0.00
source_listing_id          0.00
source_code                0.00
full_address_inherited     0.00
latest_observed_at         0.00
first_observed_at          0.00
last_observed_at           0.00
active_days                0.00
dtype: float64

In [34]:
df[df["full_address_text"].isna()]["source_code"].value_counts()

source_code
chothuephongtro    20661
mogi               14937
cafeland            7503
nhatot                 3
tromoi                 1
Name: count, dtype: int64